In [1]:

   
from tqdm import tqdm
from pathlib import Path
from os.path import join, getsize
from joblib import Parallel, delayed
from torch.utils.data import Dataset


In [13]:
from dataclasses import dataclass

@dataclass
class DataConfig:
    name: str = 'Librispeech'         # Specify corpus
    path: str = '/om/data/public/librispeech/librispeech_data/librispeech_data/' # Path to raw LibriSpeech dataset
    train_split: str = 'train-clean-100'#,'train-clean-360'] # Name of data splits to be used as training set
    dev_split: str = 'dev-clean'          # Name of data splits to be used as validation set
    bucketing: bool = True                  # Enable/Disable bucketing 
    batch_size: int = 16
config = DataConfig

In [14]:
config.train_split = [config.train_split]
config.dev_split = [config.dev_split] # Hack for getting this in list 

In [15]:
config.train_split

['train-clean-100']

In [17]:
split = config.train_split
path = config.path
file_list = []
for s in split:
    split_list = list(Path(join(path, s)).rglob("*.flac"))
    assert len(split_list) > 0, "No data found @ {}".format(join(path,s))
    file_list += split_list

In [24]:
file_list[:2]

[PosixPath('/om/data/public/librispeech/librispeech_data/librispeech_data/train-clean-100/LibriSpeech/train-clean-100/103/1240/103-1240-0000.flac'),
 PosixPath('/om/data/public/librispeech/librispeech_data/librispeech_data/train-clean-100/LibriSpeech/train-clean-100/103/1240/103-1240-0001.flac')]

In [22]:
len(file_list)

28539

In [23]:
def read_text(file):
    '''Get transcription of target wave file, 
       it's somewhat redundant for accessing each txt multiplt times,
       but it works fine with multi-thread'''
    src_file = '-'.join(file.split('-')[:-1])+'.trans.txt'
    idx = file.split('/')[-1].split('.')[0]

    with open(src_file, 'r') as fp:
        for line in fp:
            if idx == line.split(' ')[0]:
                return line[:-1].split(' ', 1)[1]
